In [1]:
import pandas as pd
import numpy as np
from prophet import Prophet
print("✅ Packages ready!")

Importing plotly failed. Interactive plots will not work.


✅ Packages ready!


In [2]:
# Download raw data (or use downloaded file)
url = "https://www.cryptodatadownload.com/cdd/Binance_BTCUSDT_d.csv"
df_raw = pd.read_csv(url, skiprows=1)  # Skip messy header row
print("Raw shape:", df_raw.shape)
print("Raw columns:", df_raw.columns.tolist())

# Clean to EXACT 6 columns needed
df_clean = df_raw[['Date', 'Open', 'High', 'Low', 'Close', 'Volume BTC']].copy()
df_clean.columns = ['date', 'open', 'high', 'low', 'close', 'volume']
df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean = df_clean.sort_values('date').tail(730)  # Last 2 years
df_clean.to_csv('ohlcv_clean.csv', index=False)

print("✅ CLEANED! Shape:", df_clean.shape)
print(df_clean.head())
print("Columns:", df_clean.columns.tolist())

Raw shape: (3072, 10)
Raw columns: ['Unix', 'Date', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Volume BTC', 'Volume USDT', 'tradecount']
✅ CLEANED! Shape: (730, 6)
          date      open      high       low     close       volume
729 2024-01-15  41732.35  43400.43  41718.05  42511.10  40269.89303
728 2024-01-16  42511.10  43578.01  42050.00  43137.95  45045.74589
727 2024-01-17  43137.94  43198.00  42200.69  42776.10  33266.21388
726 2024-01-18  42776.09  42930.00  40683.28  41327.50  43907.51641
725 2024-01-19  41327.51  42196.86  40280.00  41659.03  48342.74559
Columns: ['date', 'open', 'high', 'low', 'close', 'volume']


In [3]:
# Simple sentiment (replace with real NLP later)
dates = df_clean['date']
sentiment_scores = np.random.uniform(-0.5, 0.5, len(dates))
sentiment_df = pd.DataFrame({'date': dates, 'sentiment_score': sentiment_scores})
sentiment_df.to_csv('sentiment_data.csv', index=False)
print("✅ Sentiment created:", len(sentiment_df), "rows")

✅ Sentiment created: 730 rows


In [4]:
# Load cleaned data + sentiment (same as before)
df = pd.read_csv('ohlcv_clean.csv', parse_dates=['date']).set_index('date')
sentiment = pd.read_csv('sentiment_data.csv', parse_dates=['date']).set_index('date')
df = df.join(sentiment['sentiment_score'].fillna(0))

# Time series features (same)
df['lag1_close'] = df['close'].shift(1)
df['rolling_mean_7'] = df['close'].rolling(7).mean()
df['volatility'] = df['close'].rolling(7).std()
df['returns_pct'] = df['close'].pct_change()
df.dropna(inplace=True)

# FIXED Prophet forecasting
model_data = df.reset_index()[['date', 'close']].rename(columns={'date':'ds', 'close':'y'})
model = Prophet(daily_seasonality=True, yearly_seasonality=True)
model.fit(model_data)

# Create future dates covering BOTH historical + 30-day forecast
future_dates = model.make_future_dataframe(periods=30, freq='D')
forecast = model.predict(future_dates)

# Get forecast for HISTORICAL dates only (for signal generation)
historical_forecast = forecast[forecast['ds'].isin(df.index)]
historical_forecast.set_index('ds', inplace=True)
historical_forecast = historical_forecast[['yhat_lower', 'yhat', 'yhat_upper']].rename(
    columns={'yhat_lower':'forecast_low', 'yhat':'forecast_price', 'yhat_upper':'forecast_high'})

print("Forecast shape:", historical_forecast.shape)
print("Historical forecast dates:", historical_forecast.index[:5].tolist())

# Join forecasts to historical data ✅
df = df.join(historical_forecast, how='left')

# Generate trading signals
df['signal'] = np.where(df['forecast_price'] > df['close']*1.02, 'BUY',
                       np.where(df['forecast_price'] < df['close']*0.98, 'SELL', 'HOLD'))
df['signal_numeric'] = df['signal'].map({'BUY':1, 'SELL':-1, 'HOLD':0})

# FINAL OUTPUT
output_df = df.reset_index()
output_df.to_csv('crypto_final_output.csv', index=False)
print("🎉 FIXED PIPELINE COMPLETE!")
print(f"📊 Rows: {len(output_df)}, Columns: {len(output_df.columns)}")
print("\n📈 Sample with forecasts:")
print(output_df[['date', 'close', 'forecast_price', 'signal']].head(10))
print("\n🎯 Signals:", output_df['signal'].value_counts().to_dict())


10:50:20 - cmdstanpy - INFO - Chain [1] start processing
10:50:20 - cmdstanpy - INFO - Chain [1] done processing


Forecast shape: (724, 3)
Historical forecast dates: [Timestamp('2024-01-21 00:00:00'), Timestamp('2024-01-22 00:00:00'), Timestamp('2024-01-23 00:00:00'), Timestamp('2024-01-24 00:00:00'), Timestamp('2024-01-25 00:00:00')]
🎉 FIXED PIPELINE COMPLETE!
📊 Rows: 724, Columns: 16

📈 Sample with forecasts:
        date     close  forecast_price signal
0 2024-01-21  41580.33    41191.832025   HOLD
1 2024-01-22  39568.02    41943.261275    BUY
2 2024-01-23  39897.60    42274.178941    BUY
3 2024-01-24  40084.88    43223.191496    BUY
4 2024-01-25  39961.09    43295.780543    BUY
5 2024-01-26  41823.51    43677.143139    BUY
6 2024-01-27  42120.63    44011.484542    BUY
7 2024-01-28  42031.06    44549.330824    BUY
8 2024-01-29  43302.70    45027.859513    BUY
9 2024-01-30  42941.10    45114.021259    BUY

🎯 Signals: {'BUY': 246, 'SELL': 243, 'HOLD': 235}
